# Neural Style Transfer - Google Colab

**Notebook này đã được setup sẵn, chỉ cần chạy các cell theo thứ tự!**

## Hướng dẫn:
1. Chọn Runtime → Change runtime type → GPU (T4)
2. Chạy các cell theo thứ tự từ trên xuống
3. Upload `cuoiky.zip` và ảnh khi được yêu cầu


## Bước 1: Cài đặt packages


In [ ]:
!pip install torch torchvision pillow numpy matplotlib tqdm


In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not available! Please enable GPU: Runtime → Change runtime type → GPU")


## Bước 2: Upload Code

**Trên máy local:** Zip thư mục `cuoiky` thành `cuoiky.zip`

**Trên Colab:** Upload file `cuoiky.zip` bên dưới


In [ ]:
from google.colab import files
import zipfile
import os
import sys

# Upload file zip cuoiky.zip
print("📤 Upload file cuoiky.zip:")
uploaded = files.upload()

# Tìm file zip vừa upload
zip_filename = None
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        zip_filename = filename
        break

if zip_filename:
    # Giải nén
    print(f"📦 Extracting {zip_filename}...")
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall('/content')
    print(f"✅ Extracted {zip_filename}")
    
    # Thêm thư mục cuoiky vào Python path
    cuoiky_path = '/content/cuoiky'
    if os.path.exists(cuoiky_path):
        sys.path.insert(0, cuoiky_path)
        print(f"✅ Added {cuoiky_path} to Python path")
    else:
        print(f"⚠️ Path {cuoiky_path} not found!")
    
    # Kiểm tra file structure
    print("\n📁 File structure:")
    if os.path.exists(cuoiky_path):
        !ls -la /content/cuoiky/
        print("\n📁 Checking key files:")
        key_files = ['main.py', 'test_components.py', 'models/vgg_model.py', 'utils/losses.py']
        for file in key_files:
            file_path = f'/content/cuoiky/{file}'
            if os.path.exists(file_path):
                print(f"  ✅ {file}")
            else:
                print(f"  ❌ {file} - NOT FOUND!")
else:
    print("❌ Không tìm thấy file zip!")


## Bước 3: Upload Ảnh

Upload ảnh content và style


In [ ]:
# Tạo thư mục
!mkdir -p /content/cuoiky/data/content /content/cuoiky/data/style

# Upload content image
print("📤 Upload content image:")
content_uploaded = files.upload()
content_filename = list(content_uploaded.keys())[0]
!mv {content_filename} /content/cuoiky/data/content/content.jpg
print(f"✅ Content image saved: /content/cuoiky/data/content/content.jpg")


In [ ]:
# Upload style image
print("📤 Upload style image:")
style_uploaded = files.upload()
style_filename = list(style_uploaded.keys())[0]
!mv {style_filename} /content/cuoiky/data/style/style.jpg
print(f"✅ Style image saved: /content/cuoiky/data/style/style.jpg")


## Bước 4: Test Components (Optional)

**⚠️ Nếu gặp lỗi "File test_components.py vẫn là code CŨ":**

1. **Chạy cell "FIX: Xóa và Upload lại Code" bên dưới**
2. Upload lại `cuoiky.zip` mới (file mới nhất từ máy local)
3. Chạy lại cell test


In [ ]:
# FIX: Xóa và Upload lại Code (Chạy cell này nếu gặp lỗi code cũ)
import os
import shutil

print("🗑️ Xóa thư mục cuoiky cũ...")
if os.path.exists('/content/cuoiky'):
    shutil.rmtree('/content/cuoiky')
    print("✅ Đã xóa thư mục cũ")
else:
    print("ℹ️ Thư mục không tồn tại")

print("\n📤 Bây giờ upload lại cuoiky.zip MỚI (file mới nhất từ máy local):")
print("   1. Chạy cell Bước 2 (Upload Code) lại")
print("   2. Upload file cuoiky.zip MỚI")
print("   3. Sau đó chạy cell test bên dưới")


In [ ]:
# KIỂM TRA VÀ IMPORT - Chạy cell này sau khi upload code (Bước 2)
import os
import sys

# Kiểm tra file có tồn tại không
file_path = '/content/cuoiky/test_components.py'
if not os.path.exists(file_path):
    print(f"❌ File không tồn tại: {file_path}")
    print("\n📝 Hướng dẫn:")
    print("   1. Chạy cell Bước 2 (Upload Code) trước")
    print("   2. Upload file cuoiky.zip")
    print("   3. Sau đó chạy lại cell này")
    print("\n⚠️ Cell này sẽ không import được cho đến khi upload code xong.")
else:
    # File tồn tại, kiểm tra code mới hay cũ
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Kiểm tra xem có dòng "Testing load_image" không (code cũ)
    if 'Testing load_image' in content:
        print("❌ File test_components.py vẫn là code CŨ!")
        print("   → File cuoiky.zip bạn upload vẫn là file cũ")
        print("\n🔧 Giải pháp:")
        print("   1. Trên máy local: Zip lại thư mục cuoiky thành cuoiky.zip MỚI")
        print("   2. Chạy cell 'FIX: Xóa và Upload lại Code' ở trên")
        print("   3. Upload lại cuoiky.zip MỚI (Bước 2)")
        print("   4. Chạy lại cell này")
        print("\n⚠️ Cell này sẽ không import được cho đến khi upload code mới.")
    else:
        print("✅ File test_components.py là code MỚI - OK!")
        
        # Đảm bảo cuoiky trong path
        if '/content/cuoiky' not in sys.path:
            sys.path.insert(0, '/content/cuoiky')
        
        # Xóa cache của project
        !rm -rf /content/cuoiky/__pycache__ /content/cuoiky/*/__pycache__ 2>/dev/null || true
        
        # Import - CODE MỚI ĐÃ FIX
        try:
            from test_components import (
                test_model_loading,
                test_preprocessing,
                test_loss_functions,
                test_full_pipeline
            )
            print("✅ Imported test functions successfully")
        except Exception as e:
            print(f"❌ Import error: {e}")
            print("   → Có thể cần restart runtime: Runtime → Restart runtime")


In [ ]:
# Test từng component - CHỈ CẦN CHẠY
print("🧪 Testing components...\n")

import importlib

try:
    tc_module = importlib.import_module("test_components")
    tc_module = importlib.reload(tc_module)
    test_model_loading = tc_module.test_model_loading
    test_preprocessing = tc_module.test_preprocessing
    test_loss_functions = tc_module.test_loss_functions
    test_full_pipeline = tc_module.test_full_pipeline
except Exception as e:
    print("❌ Không thể import test_components.")
    print("   → Kiểm tra lại bước upload code (Bước 2) hoặc chạy cell FIX trước đó.")
    raise e

results = []

for name, fn in [
    ("Model Loading", test_model_loading),
    ("Preprocessing", test_preprocessing),
    ("Loss Functions", test_loss_functions),
    ("Full Pipeline", test_full_pipeline),
]:
    print(f"\n➡️ {name}")
    try:
        passed = fn()
    except KeyboardInterrupt:
        print("   ⚠️ Bị dừng giữa chừng (KeyboardInterrupt). Chạy lại cell để hoàn tất.")
        raise
    except Exception as err:
        print(f"   ❌ Lỗi: {err}")
        passed = False
    results.append((name, passed))

print("\n📊 Kết quả:")
for name, passed in results:
    status = "✅ PASSED" if passed else "❌ FAILED"
    print(f"- {name}: {status}")

if all(passed for _, passed in results):
    print("\n🎉 All tests completed successfully!")
else:
    print("\n⚠️ Một số test lỗi, xem log ở trên để xử lý.")


## Bước 5: Train Neural Style Transfer

**⚠️ QUAN TRỌNG:** Đảm bảo đã upload ảnh ở Bước 3 trước khi train!

Chạy training với ảnh đã upload


In [ ]:
# KIỂM TRA ẢNH TRƯỚC KHI TRAIN - CHẠY CELL NÀY TRƯỚC!
import os

content_path = "/content/cuoiky/data/content/content.jpg"
style_path = "/content/cuoiky/data/style/style.jpg"

print("🔍 Kiểm tra file ảnh...")
print(f"   Content path: {content_path}")
print(f"   Style path: {style_path}\n")

# Kiểm tra content image
if not os.path.exists(content_path):
    print(f"❌ Content image KHÔNG TỒN TẠI: {content_path}")
    print("\n📝 Hướng dẫn:")
    print("   1. Chạy cell Bước 3 - Upload Content Image (Cell 7)")
    print("   2. Upload file ảnh content của bạn")
    print("   3. Sau đó chạy lại cell này")
    raise FileNotFoundError(f"Content image not found: {content_path}")
else:
    size = os.path.getsize(content_path)
    print(f"✅ Content image: {content_path}")
    print(f"   File size: {size:,} bytes ({size/1024/1024:.2f} MB)")

# Kiểm tra style image
if not os.path.exists(style_path):
    print(f"\n❌ Style image KHÔNG TỒN TẠI: {style_path}")
    print("\n📝 Hướng dẫn:")
    print("   1. Chạy cell Bước 3 - Upload Style Image (Cell 8)")
    print("   2. Upload file ảnh style của bạn")
    print("   3. Sau đó chạy lại cell này")
    raise FileNotFoundError(f"Style image not found: {style_path}")
else:
    size = os.path.getsize(style_path)
    print(f"✅ Style image: {style_path}")
    print(f"   File size: {size:,} bytes ({size/1024/1024:.2f} MB)")

print("\n✅ Tất cả ảnh đã sẵn sàng!")
print("🚀 Bắt đầu training...\n")

from main import train_neural_style_transfer, STYLE_PRESETS

preset_name = "baseline"  # 'baseline', 'vangogh_strong', 'vangogh_extreme'
preset_params = STYLE_PRESETS[preset_name].copy()
output_path = f"/content/cuoiky/output_{preset_name}.jpg"

print(f"\n🎚 Đang dùng preset: {preset_name}")
for key, value in preset_params.items():
    if key != "style_layer_weights":
        print(f"   {key}: {value}")
if preset_params.get("style_layer_weights"):
        print(f"   style_layer_weights: {preset_params['style_layer_weights']}")

generated_image, losses = train_neural_style_transfer(
    content_path=content_path,
    style_path=style_path,
    output_path=output_path,
    device='cuda',
    **preset_params
)

print(f"\n✅ Training hoàn tất! Kết quả lưu tại: {output_path}")


## Bước 6: Download Kết Quả

Download ảnh kết quả về máy


In [ ]:
from google.colab import files
import glob

# Download kết quả
files.download('/content/cuoiky/output.jpg')

# Download loss plot nếu có
if os.path.exists('/content/cuoiky/output_losses.png'):
    files.download('/content/cuoiky/output_losses.png')

# Download checkpoints nếu có
for checkpoint in glob.glob('/content/cuoiky/*_iter_*.jpg'):
    files.download(checkpoint)
    print(f"✅ Downloaded: {checkpoint}")
